# Concept: Using OpenAI Python Client with Multiple LLM Providers

Welcome to this architectural deep-dive. In this lesson, we will explore the **Unified Client Pattern** using the OpenAI SDK as our standard interface for various cloud providers.

### Why this matters?
Modern AI architectures must be **resilient and cost-effective**. By using the OpenAI-compatible interface, you can switch between providers like DeepSeek (for cost) or xAI Grok (for real-time information and reasoning) without rewriting your core business logic.

## 1. Setup

Ensure you have the latest OpenAI library installed and your `.env` file contains keys for OpenAI, DeepSeek, and xAI.

In [5]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv() # Load your API keys: OPENAI_API_KEY, DEEPSEEK_API_KEY, XAI_API_KEY

True

## 2. Standard Implementation (OpenAI)

The baseline using GPT-4o-mini.

In [6]:
client_openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

response = client_openai.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": "Explain latency in 10 words."}]
)

print(f"OpenAI: {response.choices[0].message.content}")

OpenAI: Latency is the delay before data transmission begins or completes.


## 3. Cost Optimization (DeepSeek)

DeepSeek provides an OpenAI-compatible API that is significantly cheaper for many use cases.

In [7]:
client_deepseek = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

response = client_deepseek.chat.completions.create(
    model="deepseek-chat",
    messages=[{"role": "user", "content": "Explain latency in 10 words."}]
)

print(f"DeepSeek: {response.choices[0].message.content}")

DeepSeek: Delay between action and response in a system.


## 4. Real-time Knowledge (xAI Grok)

xAI's Grok models are accessible via an OpenAI-compatible endpoint, making integration seamless.

In [8]:
client_xai = OpenAI(
    api_key=os.getenv("XAI_API_KEY"),
    base_url="https://api.x.ai/v1"
)

response = client_xai.chat.completions.create(
    model="grok-4-1-fast-non-reasoning",
    messages=[{"role": "user", "content": "Explain latency in 10 words."}]
)

print(f"DeepSeek: {response.choices[0].message.content}")

DeepSeek: Delay between input and output in networks or systems.


## 5. Architectural Summary: The Provider Factory

In a real-world application, we wrap this in a factory function to maintain a single interface.

In [9]:
def get_client(provider):
    """
    Resolves the correct OpenAI client and model based on the provider name.
    """
    config = {
        "openai": {"api_key": os.getenv("OPENAI_API_KEY"), "base_url": None, "model": "gpt-4o-mini"},
        "deepseek": {"api_key": os.getenv("DEEPSEEK_API_KEY"), "base_url": "https://api.deepseek.com", "model": "deepseek-chat"},
        "xai": {"api_key": os.getenv("XAI_API_KEY"), "base_url": "https://api.x.ai/v1", "model": "grok-4-1-fast-non-reasoning"}
    }
    
    selected = config.get(provider.lower())
    if not selected:
        raise ValueError(f"Provider {provider} not supported.")

    return OpenAI(api_key=selected["api_key"], base_url=selected["base_url"]), selected["model"]

## 6. Dynamic Provider Switching (The Switchboard)

This final function acts as a unified entry point. You can now call any LLM for the same query by simply changing the provider name.

In [10]:
def chat_with_llm(provider_name, user_query):
    """
    A unified entry point to switch LLMs dynamically.
    """
    client, model = get_client(provider_name)
    
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": user_query}]
    )
    
    return response.choices[0].message.content

# Architects can now execute the same query across different brains:
query = "What is the main benefit of using a unified API client?"

print(f"--- Executing via OpenAI ---\n{chat_with_llm('openai', query)}\n")
print(f"--- Executing via DeepSeek ---\n{chat_with_llm('deepseek', query)}\n")
print(f"--- Executing via xAI ---\n{chat_with_llm('xai', query)}\n")

--- Executing via OpenAI ---
The main benefit of using a unified API client is that it simplifies the interaction with multiple APIs by providing a consistent and cohesive interface. This has several advantages:

1. **Simplified Development**: Developers can interact with various APIs using a single set of methods and structures, reducing the need to learn and manage different libraries or SDKs for each API.

2. **Reduced Complexity**: A unified API client abstracts the complexities of individual APIs, including authentication, error handling, and data formatting, allowing developers to focus on building their applications rather than dealing with integration challenges.

3. **Improved Maintenance**: With a unified interface, updating or maintaining the integration with various APIs becomes easier. Changes in one API don’t necessarily require significant changes to the overall codebase.

4. **Consistent Error Handling**: A unified API client can standardize error handling across differ